In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
# Installing local embedding libraries to bypass OpenAI quota issue
!pip install -q --no-warn-conflicts chromadb langchain-chroma langchain-community langchain-text-splitters sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 636.6 kB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 42.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 53.4 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 43.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 11.3 MB/s eta 0:00:00
   ━━

In [2]:
import os
import numpy as np
import chromadb
from chromadb.utils import embedding_functions
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("✓ Local setup ready. No OpenAI API Key needed now!")

/tmp/ipykernel_57/1666337794.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


✓ Local setup ready. No OpenAI API Key needed now!


In [3]:
import os

# Create directory
os.makedirs('company_docs', exist_ok=True)

# Generate sample files
sample_docs = {
    "vacation_policy.txt": "Employees get 20 days of paid time off (PTO) annually. All vacation requests must be submitted two weeks in advance.",
    "wfh_policy.txt": "Our remote work and WFH guidelines allow employees to work from home up to 3 days a week with manager approval.",
    "maternity_policy.txt": "The parental leave policy provides 12 weeks of fully paid maternity leave for new mothers."
}

for filename, content in sample_docs.items():
    with open(f"company_docs/{filename}", "w") as f:
        f.write(content)

print("✓ Setup complete: Sample documents generated in 'company_docs/' folder.")

✓ Setup complete: Sample documents generated in 'company_docs/' folder.


In [4]:
from sentence_transformers import SentenceTransformer

# Load a completely free, open-source embedding model
local_model = SentenceTransformer('all-MiniLM-L6-v2')

def get_embedding(text):
    """Generate embedding for text using local Hugging Face model. Returns: List of numbers."""
    embedding = local_model.encode(text)
    return embedding.tolist()

# Test the embedding function
text = 'vacation policy'
embedding = get_embedding(text)
print(f'Embedding length: {len(embedding)} (Using local 384-dim model)')
print(f'First 5 values: {embedding[:5]}')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding length: 384 (Using local 384-dim model)
First 5 values: [0.04388044774532318, 0.05842041224241257, 0.08754625916481018, -0.0002513062208890915, 0.03657166287302971]


In [5]:
def cosine_similarity(vec1, vec2):
    """Calculate cosine similarity between two vectors."""
    v1 = np.array(vec1)
    v2 = np.array(vec2)
    
    dot_product = np.dot(v1, v2)
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    
    return dot_product / (norm1 * norm2)

# Test with similar and dissimilar phrases
phrases = ['vacation policy', 'time off rules', 'PTO guidelines', 'dress code requirements']
embeddings = [get_embedding(p) for p in phrases]

base_embedding = embeddings[0]
print(f'Comparing "{phrases[0]}" with:\n')

for i, phrase in enumerate(phrases[1:], start=1):
    similarity = cosine_similarity(base_embedding, embeddings[i])
    print(f'{phrase:30} Similarity: {similarity:.4f}')

Comparing "vacation policy" with:

time off rules                 Similarity: 0.2653
PTO guidelines                 Similarity: 0.1812
dress code requirements        Similarity: 0.1095


In [6]:
import chromadb
from chromadb.utils import embedding_functions

# Create ChromaDB persistent client
chroma_client = chromadb.PersistentClient(path='./chromadb')

# Use ChromaDB's built-in default embedding function (Sentence Transformers)
# This works 100% locally and does NOT require any API key!
default_ef = embedding_functions.DefaultEmbeddingFunction()

# Get or create collection
collection = chroma_client.get_or_create_collection(
    name='company_docs',
    embedding_function=default_ef,
    metadata={'description': 'Company policy documents'}
)

print(f'Collection Name: {collection.name}')
print(f'Current Document Count: {collection.count()}')

Collection Name: company_docs
Current Document Count: 0


In [8]:
# Load text documents from the folder
loader = DirectoryLoader('company_docs/', glob='*.txt', loader_cls=TextLoader)
documents = loader.load()

# Split into chunks of 500 characters
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

# Add to ChromaDB if empty
if collection.count() == 0:
    collection.add(
        documents=[chunk.page_content for chunk in chunks],
        ids=[f'doc_{i}' for i in range(len(chunks))],
        metadatas=[{'source': chunk.metadata.get('source', 'unknown')} for chunk in chunks]
    )
    print(f'✓ Successfully added {len(chunks)} chunks to ChromaDB.')
else:
    print(f'Collection already contains {collection.count()} documents. Skipping insertion.')

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:06<00:00, 12.6MiB/s]


✓ Successfully added 3 chunks to ChromaDB.


In [9]:
def vector_search(query, n_results=3):
    """Search ChromaDB collection using semantic similarity."""
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    return results

# Test queries
test_queries = ['time off policy', 'WFH guidelines', 'maternity leave']

for query in test_queries:
    print(f'\n{"="*60}')
    print(f'Query: {query}')
    print(f'{"="*60}')
    
    results = vector_search(query, n_results=2)
    
    for i, doc in enumerate(results['documents'][0]):
        distance = results['distances'][0][i]
        print(f'\nResult {i+1} (Distance/Score: {distance:.4f}):')
        print(doc[:200] + '...')


Query: time off policy

Result 1 (Distance/Score: 1.0252):
Employees get 20 days of paid time off (PTO) annually. All vacation requests must be submitted two weeks in advance....

Result 2 (Distance/Score: 1.3234):
The parental leave policy provides 12 weeks of fully paid maternity leave for new mothers....

Query: WFH guidelines

Result 1 (Distance/Score: 1.2676):
Our remote work and WFH guidelines allow employees to work from home up to 3 days a week with manager approval....

Result 2 (Distance/Score: 1.8423):
The parental leave policy provides 12 weeks of fully paid maternity leave for new mothers....

Query: maternity leave

Result 1 (Distance/Score: 0.6504):
The parental leave policy provides 12 weeks of fully paid maternity leave for new mothers....

Result 2 (Distance/Score: 1.5459):
Our remote work and WFH guidelines allow employees to work from home up to 3 days a week with manager approval....


In [11]:
# Install required package
!pip install -q langchain-openai langchain-community langchain

# Import
from langchain_openai import ChatOpenAI
import os

# Set API Key
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# Initialize LLM
try:
    llm = ChatOpenAI(
        model="gpt-3.5-turbo",
        temperature=0
    )
    print("LLM Initialized Successfully!")

except Exception as e:
    print("Error:", e)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 1.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 7.1 MB/s eta 0:00:0000:0100:01
LLM Initialized Successfully!


In [12]:
def keyword_search(query, chunks, top_k=3):
    """Simple keyword frequency counter."""
    scored = []
    query_lower = query.lower()
    words = query_lower.split()
    
    for chunk in chunks:
        score = sum(chunk.page_content.lower().count(word) for word in words)
        if score > 0:
            scored.append((score, chunk))
            
    scored.sort(key=lambda x: x[0], reverse=True)
    return [c for s, c in scored[:top_k]]

# Run Test
query = 'PTO policy'

print('--- KEYWORD SEARCH ---')
kw_results = keyword_search(query, chunks, top_k=2)
print(f'Found: {len(kw_results)} results')

print('\n--- SEMANTIC SEARCH ---')
sem_results = vector_search(query, n_results=2)
print(f'Found: {len(sem_results["documents"][0])} results')
if len(sem_results["documents"][0]) > 0:
    print(f'Top result: {sem_results["documents"][0][0][:120]}...')

--- KEYWORD SEARCH ---
Found: 2 results

--- SEMANTIC SEARCH ---
Found: 2 results
Top result: Employees get 20 days of paid time off (PTO) annually. All vacation requests must be submitted two weeks in advance....


In [13]:
# Installing Streamlit and local tunnel utility for Kaggle web preview
!pip install -q streamlit
!npm install -q -g localtunnel
print("✓ Streamlit and Localtunnel successfully installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 23.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 49.9 MB/s eta 0:00:0000:010:01m
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
added 22 packages in 2s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼npm notice
npm notice New major version of npm available! 10.8.2 -> 11.15.0
npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.15.0
npm notice To update run: npm install -g npm@11.15.0
npm notice
⠼✓ Streamlit and Localtunnel successfully installed!


In [14]:
%%writefile app.py
import streamlit as st
import os
import chromadb
from chromadb.utils import embedding_functions
from langchain_openai import ChatOpenAI

st.set_page_config(page_title='AI Assistant', page_icon='🤖', layout='wide')

try:
    @st.cache_resource
    def init_chromadb_with_dummy_data():
        """Initializes ChromaDB and inserts complete policy text directly to guarantee answers."""
        client = chromadb.PersistentClient(path='./chromadb')
        default_ef = embedding_functions.DefaultEmbeddingFunction()
        
        collection = client.get_or_create_collection(
            name='company_docs',
            embedding_function=default_ef
        )
        
        # Injecting direct text data so the system is never empty
        if collection.count() == 0:
            policies = [
                "Remote Work Guidelines: Employees can work from home up to 3 days per week with manager approval. Core working hours for remote tracking are 10:00 AM to 4:00 PM. Permanent work from home requires HR executive escalation.",
                "Vacation and Time Off Policies: All full-time employees are entitled to 25 days of annual vacation leave per calendar year. Emergency time off must be logged via the HR portal at least 2 hours before shifts.",
                "Parental Leave Benefits: The company provides 16 weeks of fully paid maternity leave for birth mothers and 4 weeks of fully paid paternity leave for secondary caregivers. Benefits apply immediately after probation."
            ]
            collection.add(
                documents=policies,
                ids=[f"policy_{i}" for i in range(len(policies))],
                metadatas=[{"source": "hr_handbook"} for _ in policies]
            )
        return collection

    @st.cache_resource
    def init_llm():
        # Safeguard dummy key configuration to prevent API crashes
        if not os.environ.get("OPENAI_API_KEY") or os.environ.get("OPENAI_API_KEY") == "sk-proj-YOUR_OPENAI_KEY_HERE":
            os.environ["OPENAI_API_KEY"] = "dummy-key-for-local-retrieval"
        return ChatOpenAI(model='gpt-3.5-turbo', temperature=0)

    collection = init_chromadb_with_dummy_data()
    llm = init_llm()

    def get_rag_response(query, n_results=1):
        try:
            results = collection.query(query_texts=[query], n_results=n_results)
            if not results['documents'] or not results['documents'][0]:
                return 'No relevant information found in documents.'
            
            context = '\n'.join(results['documents'][0])
            
            # Since OpenAI key is local/dummy, cleanly return the exact matched chunk
            return f"🤖 **[Context Retrieved From ChromaDB Successfully!]**\n\n{context}"
        except Exception as e:
            return f'Error: {str(e)}'

    # Application UI Configuration
    st.title('🏢 Company Knowledge Assistant')
    st.markdown('Ask me anything about company policies!')

    if 'messages' not in st.session_state:
        st.session_state.messages = []

    with st.sidebar:
        st.header('ℹ️ About')
        st.markdown("Powered by local ChromaDB Vector Search.")
        st.divider()
        st.metric('Documents Indexed', collection.count())
        st.metric('Messages in Chat', len(st.session_state.messages))
        st.divider()
        if st.button('Clear Chat History'):
            st.session_state.messages = []
            st.rerun()

    if len(st.session_state.messages) == 0:
        with st.chat_message('assistant'):
            st.write("Hi! I'm your company knowledge assistant. 👋 Ask me a question to get started.")

    for message in st.session_state.messages:
        with st.chat_message(message['role']):
            st.write(message['content'])

    if prompt := st.chat_input('Ask a question...'):
        st.session_state.messages.append({'role': 'user', 'content': prompt})
        with st.chat_message('user'):
            st.write(prompt)
            
        with st.chat_message('assistant'):
            with st.spinner('Searching database...'):
                response = get_rag_response(prompt)
                st.write(response)
        st.session_state.messages.append({'role': 'assistant', 'content': response})

except Exception as e:
    st.error(f'System Error: {str(e)}')
    st.stop()

Writing app.py


In [15]:
# Extract the public IP to use as the password tunnel gateway
!curl ipv4.icanhazip.com

34.80.159.210


In [ ]:
import subprocess
import threading
import time

print("🧹 Cleaning older background processes...")
!pkill -f streamlit
!pkill -f localtunnel
!pkill -f ngrok
!pkill -f ssh

print("🚀 Starting Streamlit background server...")
def run_app():
    subprocess.Popen("streamlit run app.py --server.port=8501 --server.address=0.0.0.0", shell=True)

threading.Thread(target=run_app, daemon=True).start()
time.sleep(5)

print("\n🌐 Generating dynamic clean public link via Serveo...")
print("Click the link below when it appears. Ignore any 'warning' or click 'Continue' if asked.")
print("="*60)

# This exposes port 8501 globally without requiring any login/token/password
!ssh -o StrictHostKeyChecking=no -R 80:localhost:8501 serveo.net

🧹 Cleaning older background processes...
🚀 Starting Streamlit background server...




2026-05-25 08:14:58.281 Uvicorn server started on 0.0.0.0:8501



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://34.80.159.210:8501


🌐 Generating dynamic clean public link via Serveo...
Click the link below when it appears. Ignore any 'warning' or click 'Continue' if asked.
Forwarding HTTP traffic from https://2318df4efe54c0b3-34-80-159-210.serveousercontent.com
Tip (1): Create an account to reserve names. Pro removes the warning page: https://console.serveo.net/settings?n=1&src=ssh_nudge&v=B
